# 04buildincome: household income for steps 4 and 5

PF7, "Kita: Malaysia in 3.2 million dots". One small file, `docs/data/income.json`, from the Household Income Survey (HIES) 2022 published on OpenDOSM:

1. **By state, percentiles 1 to 100**: the income range of each hundredth of households. The page uses it for "Where do you stand?": a reader's income is placed inside its state's percentiles.
2. **By district**: median and mean income, Gini and poverty rate, and the district's bounding box (so the map can fly there), for the district picker and the beeswarm of step 4.

All figures are gross monthly household income in ringgit (RM), 2022. Percentiles are published by state only, so the reader is compared with households in their state; the district median is shown beside it.

## 0. Settings

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd()                                   # <- run from the PF7 folder
RAW = ROOT / "data" / "raw"
WEB = ROOT / "docs" / "data"
WEB.mkdir(parents=True, exist_ok=True)

YEAR = 2022                                         # <- survey year (matches Census 2020 districts)

## 1. State percentiles

Each state has 100 rows (percentile 1 to 100) with the minimum, median, mean and maximum income of the households in that hundredth. The lowest minimum and the highest maximum are not published; the page uses the median there.

In [ ]:
p = pd.read_parquet(RAW / "hies_state_percentile.parquet")
p = p[pd.to_datetime(p["date"]).dt.year == YEAR]
wide = p.pivot_table(index=["state", "percentile"], columns="variable", values="income").reset_index()
print(wide["state"].nunique(), "states;", len(wide), "rows")
print("empty cells:", wide[["minimum", "median", "maximum"]].isna().sum().to_dict())
wide["lo"] = wide["minimum"].fillna(wide["median"])
wide["hi"] = wide["maximum"].fillna(wide["median"])
print(wide[wide["state"] == "Selangor"].set_index("percentile").loc[[1, 10, 50, 90, 100], ["lo", "median", "hi"]].round(0).to_string())

In [ ]:
order_ok = []
for st, d in wide.groupby("state"):
    d = d.sort_values("percentile")
    order_ok.append(bool((np.diff(d["median"].to_numpy()) >= 0).all()))
print("medians rise with the percentile in every state:", all(order_ok))

## 2. Districts

In [ ]:
dist = pd.read_parquet(RAW / "hies_district.parquet")
dist = dist[pd.to_datetime(dist["date"]).dt.year == YEAR].copy()
print(len(dist), "districts")
print(dist[["income_median", "income_mean", "gini", "poverty"]].describe().round(2).to_string())
print("lowest median:", dist.nsmallest(3, "income_median")[["state", "district", "income_median"]].to_dict("records"))
print("highest median:", dist.nlargest(3, "income_median")[["state", "district", "income_median"]].to_dict("records"))

## 3. Write income.json

In [ ]:
states = {}
for st, d in wide.groupby("state"):
    d = d.sort_values("percentile")
    states[st] = {"lo": d["lo"].round(0).astype(int).tolist(), "med": d["median"].round(0).astype(int).tolist(), "hi": d["hi"].round(0).astype(int).tolist()}
import geopandas as gpd
shapes = gpd.read_file(RAW / "administrative_2_district.geojson")
shapes["k"] = shapes["state"].str.lower() + "|" + shapes["district"].str.lower()
box = shapes.set_index("k").geometry.bounds.round(4)
dist["k"] = dist["state"].str.lower() + "|" + dist["district"].str.lower()
print("districts without a shape:", (~dist["k"].isin(box.index)).sum())
districts = []
for r in dist.sort_values(["state", "district"]).itertuples():
    b = box.loc[r.k]
    districts.append({"state": r.state, "district": r.district, "median": int(round(r.income_median)), "mean": int(round(r.income_mean)), "gini": round(float(r.gini), 3), "poverty": round(float(r.poverty), 1), "bbox": [b.minx, b.miny, b.maxx, b.maxy]})
income = {"year": YEAR, "unit": "RM per month, gross household income", "source": "DOSM, Household Income Survey 2022 (OpenDOSM)", "states": states, "districts": districts}
(WEB / "income.json").write_text(json.dumps(income))
print(f"income.json {(WEB / 'income.json').stat().st_size / 1e3:.0f} kB, {len(states)} states, {len(districts)} districts")

___

**END**